In [8]:
# Cell 1: setup imports and create problem and model
import sys, os
#sys.path.append(os.path.join(os.path.dirname(__file__),  'src'))  # adjust if file path different

import torch
import numpy as np
import matplotlib.pyplot as plt

from src.pde_problems import PlateWithHole
from src.pinn import PINN

# problem parameters
a = 0.2   # hole radius (must match training)
b = 1.0

# create problem instance
problem = PlateWithHole(a=a, b=b, n_collocation=3600, n_boundary=1300)

# try to load a saved model if you have it, otherwise create a fresh model
model_path = os.path.join('results', 'plate_with_hole', 'plate_hole_model.pth')
if os.path.exists(model_path):
    try:
        model = PINN.load_model(model_path, hidden_layers=[20,20,20,20], activation='tanh')
        print("Loaded model from", model_path)
    except Exception as e:
        print("Failed loading checkpoint, creating new model:", e)
        model = PINN(input_dim=2, hidden_layers=[20,20,20,20], output_dim=1, activation='tanh')
else:
    print("No checkpoint found. Creating an untrained model.")
    model = PINN(input_dim=2, hidden_layers=[20,20,20,20], output_dim=1, activation='tanh')

model.eval()

Loaded model from results/plate_with_hole/plate_hole_model.pth


PINN(
  (network): Sequential(
    (0): Linear(in_features=2, out_features=20, bias=True)
    (1): Tanh()
    (2): Linear(in_features=20, out_features=20, bias=True)
    (3): Tanh()
    (4): Linear(in_features=20, out_features=20, bias=True)
    (5): Tanh()
    (6): Linear(in_features=20, out_features=20, bias=True)
    (7): Tanh()
    (8): Linear(in_features=20, out_features=1, bias=True)
  )
)

In [9]:
# Cell 2: helper to compute gradients robustly
def safe_grad(y, x):
    g = torch.autograd.grad(y, x, grad_outputs=torch.ones_like(y), create_graph=True, allow_unused=True)[0]
    if g is None:
        return torch.zeros_like(y)
    return g

In [10]:
# Cell 3: compute residual (u_xx + u_yy - source) on a grid and plot
n = 200
xs = torch.linspace(-b, b, n)
ys = torch.linspace(-b, b, n)
X, Y = torch.meshgrid(xs, ys, indexing='xy')
pts_all = torch.stack([X.flatten(), Y.flatten()], dim=1).float()

# mask outside hole
r_all = torch.sqrt(pts_all[:,0:1]**2 + pts_all[:,1:2]**2)
mask = (r_all >= a).squeeze()

# select masked points and enable grad for inputs
pts = pts_all[mask].clone().requires_grad_(True)

# IMPORTANT: use model(pts) (not model.predict) so autograd sees the graph
u = model(pts)           # shape (M,1)
u = u.reshape(-1, 1)

# compute derivatives
ux  = safe_grad(u, pts[:,0:1])
uy  = safe_grad(u, pts[:,1:2])
uxx = safe_grad(ux, pts[:,0:1])
uyy = safe_grad(uy, pts[:,1:2])

# source term (avoid r==0)
r = torch.sqrt(pts[:,0:1]**2 + pts[:,1:2]**2)
r = torch.where(r == 0.0, torch.ones_like(r) * 1e-8, r)
source = 4.0 - 2.0 * a / r

residual = (uxx + uyy - source).detach().cpu().numpy().flatten()

# put residual back into full grid (NaN inside hole)
residual_all = np.full((n*n,), np.nan, dtype=float)
residual_all[mask.cpu().numpy()] = residual
residual_grid = residual_all.reshape(n, n)

# plot
plt.figure(figsize=(6,5))
plt.contourf(X.numpy(), Y.numpy(), residual_grid, levels=60, cmap='RdBu_r')
plt.colorbar(label='PDE residual (u_xx + u_yy - source)')
plt.title('PDE residual (masked hole)')
plt.xlabel('x'); plt.ylabel('y')
plt.show()

# optional quick stats
valid = residual[~np.isnan(residual)]
print('Residual mean, max_abs, std:', np.mean(valid), np.max(np.abs(valid)), np.std(valid))

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn